In [15]:
# Import the NumPy library for numerical operations, commonly used for array manipulation.
import numpy as np
# Import the TensorFlow library, an open-source machine learning framework.
import tensorflow as tf
# Import Tokenizer for text processing, specifically for converting text to sequences of numbers.
from tensorflow.keras.preprocessing.text import Tokenizer
# Import pad_sequences for ensuring all input sequences to the model have the same length.
from tensorflow.keras.preprocessing.sequence import pad_sequences
# Import Model to create a Keras functional API model, allowing flexible model architectures.
from tensorflow.keras.models import Model
# Import various Keras layers: Input for defining model input, Embedding for word embeddings,
# SimpleRNN for the recurrent neural network layer, and Dense for fully connected layers.
from tensorflow.keras.layers import Input, Embedding, SimpleRNN, Dense

### 1. Define Sentences and Labels

In [16]:
# Define a list of sentences, representing positive and negative sentiments for training.
sentences = [
 "I love this product",
 "This movie made me smile",
 "Service was friendly and quick",
 "Today felt bright and happy",
 "This is the best day",
 "Absolutely fantastic experience",
 "I enjoyed every single moment",
 "Great job, well done",
 "The food tasted delicious",
 "Totally recommend to everyone",
 "Very satisfied with results",
 "This worked better than expected",
 "Amazing quality and value",
 "Such a pleasant surprise",
 "I feel positive about this",
 "I hate this product",
 "This movie bored me",
 "Service was rude and slow",
 "Today was cold and lonely",
 "This is the worst day",
 "Terrible experience overall",
 "I regret buying this",
 "Very disappointed with results",
 "The food tasted awful",
 "Do not recommend this",
 "It broke after one use",
 "Not worth the money",
 "Utterly frustrating and annoying",
 "I feel negative about this",
 "Such a waste of time",
]

### 2. Tokenization and Padding

In [17]:
# Define the `labels` for the sentences: 1 for positive sentiment, 0 for negative sentiment.
labels = [1]*15 + [0]*15
# Convert the list of labels to a NumPy array for efficient processing with TensorFlow.
labels = np.array(labels)
# Display the `labels` array to verify its content.
labels

array([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0])

In [11]:
# Define the `labels` for the sentences: 1 for positive sentiment, 0 for negative sentiment.
labels = [1]*15 + [0]*15
# Convert the list of labels to a NumPy array for efficient processing with TensorFlow.
labels = np.array(labels)
# Display the `labels` array to verify its content.
labels

array([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0])

In [18]:
# Define the maximum size of the vocabulary for the tokenizer.
vocab_size = 2000
# Initialize the Tokenizer. `num_words` limits the vocabulary size, `oov_token` handles out-of-vocabulary words.
tok  = Tokenizer(num_words = vocab_size,oov_token = "")
# Fit the tokenizer on the `sentences` to build the vocabulary.
tok.fit_on_texts(sentences)
# Convert the sentences into sequences of integers based on the tokenizer's vocabulary.
seqs = tok.texts_to_sequences(sentences)
# Determine the maximum length among all generated sequences for padding.
maxlen = max(len(s) for s in seqs)
# Pad the sequences to ensure they all have the same length (`maxlen`). 'post' padding adds zeros at the end.
X = pad_sequences(seqs,maxlen = maxlen,padding = 'post')
# Assign the NumPy array of labels to `y`, preparing it for model training.
y = labels

In [12]:
# Define the maximum size of the vocabulary for the tokenizer.
vocab_size = 2000
# Initialize the Tokenizer. `num_words` limits the vocabulary size, `oov_token` handles out-of-vocabulary words.
tok  = Tokenizer(num_words = vocab_size,oov_token = "")
# Fit the tokenizer on the `sentences` to build the vocabulary.
tok.fit_on_texts(sentences)
# Convert the sentences into sequences of integers based on the tokenizer's vocabulary.
seqs = tok.texts_to_sequences(sentences)
# Determine the maximum length among all generated sequences for padding.
maxlen = max(len(s) for s in seqs)
# Pad the sequences to ensure they all have the same length (`maxlen`). 'post' padding adds zeros at the end.
X = pad_sequences(seqs,maxlen = maxlen,padding = 'post')
# Assign the NumPy array of labels to `y`, preparing it for model training.
y = labels

In [19]:
# Display the first processed sequence (sentence) in numerical format after tokenization and padding.
X[0]

array([ 3, 26,  2,  7,  0], dtype=int32)

### 3. Model Parameters

In [20]:
# Define the dimensionality of the word embeddings. This will be the size of the vector representation for each word.
embed_dim = 16
# Define the number of units (neurons) in the SimpleRNN layer. This determines the capacity of the RNN to learn.
rnn_units = 8

### 4. Build and Compile the Simple RNN Model

In [21]:
# Define the input layer of the model: expects sequences of integers with length `maxlen`.
inp = Input(shape= (maxlen,),dtype="int32",name = 'input')
# Add an Embedding layer: converts integer-encoded words into dense vectors of size `embed_dim`.
# `mask_zero=True` handles padded zeros by making them ignorable during sequence processing.
x = Embedding(input_dim=vocab_size,output_dim=embed_dim,mask_zero=True,name = 'embed')(inp)
# Add a SimpleRNN layer: processes the sequence of embeddings.
# `return_sequences=False` means it returns only the final hidden state, suitable for classification.
rnn = SimpleRNN(units = rnn_units, return_sequences = False,return_state= False,name = 'simple_rnn')
# Pass the embedded sequences through the SimpleRNN layer.
x_last = rnn(x)
# Add a Dense output layer with a single unit and sigmoid activation for binary classification.
out = Dense(1,activation='sigmoid',name = 'out')(x_last)
# Create the Keras Model by specifying its inputs and outputs.
model = Model(inputs =inp,outputs = out)
# Compile the model:
# `optimizer='adam'` is a popular choice for gradient descent optimization.
# `loss='binary_crossentropy'` is suitable for binary classification tasks.
# `metrics=['accuracy']` tracks the classification accuracy during training.
model.compile(optimizer='adam',loss = 'binary_crossentropy',metrics=['accuracy'])
# Print a summary of the model's architecture, including layer types, output shapes, and number of parameters.
model.summary()

Model: "functional_5"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input (InputLayer)  │ (None, 5)         │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embed (Embedding)   │ (None, 5, 16)     │     32,000 │ input[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_3         │ (None, 5)         │          0 │ input[0][0]       │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ simple_rnn          │ (None, 8)         │        200 │ embed[0][0],      │
│ (SimpleRNN)         │                   │            │ not_equal_3[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ out (Dense)         │ (None, 1)         │          9 │ simple_rnn[0][0]  │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 32,209 (125.82 KB)

 Trainable params: 32,209 (125.82 KB)

 Non-trainable params: 0 (0.00 B)

### 5. Train the Model

In [22]:
# Train the model using the prepared data X (input sequences) and y (labels).
# `epochs=25` specifies the number of full passes over the training dataset.
# `batch_size=8` defines the number of samples per gradient update.
# `verbose=1` displays a progress bar during training.
model.fit(X,y,epochs= 25,batch_size = 8,verbose = 1)

Epoch 1/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - accuracy: 0.5000 - loss: 0.7038
Epoch 2/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.5333 - loss: 0.6845
Epoch 3/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.6667 - loss: 0.6678
Epoch 4/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.7667 - loss: 0.6515
Epoch 5/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.8667 - loss: 0.6353
Epoch 6/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.9333 - loss: 0.6163
Epoch 7/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.9333 - loss: 0.5967
Epoch 8/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.9333 - loss: 0.5761
Epoch 9/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.9667 - loss: 0.5536
Epoch 10/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.9667 - loss: 0.5312
Epoch 11/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.9667 - loss: 0.5054
Epoch 12/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.9667 - loss: 0.4805
E

### 6. Inspecting Intermediate States

This section demonstrates how to create a new Keras Model that takes the same input as the original model but outputs the embedding layer and the hidden states of the SimpleRNN layer. This allows for inspection of the internal representations learned by the model at different stages.

Here's a new Keras Model that takes the same input as the original model but outputs the embedding layer and the hidden states of the SimpleRNN layer.

In [23]:
# Create a new Keras Model that takes the same input as the original `model`.
# It outputs the embedding layer's output and the SimpleRNN layer's output (hidden states).
intermediate_model = Model(inputs=model.inputs, outputs=[model.get_layer('embed').output, model.get_layer('simple_rnn').output])

In [13]:
# Create a new Keras Model that takes the same input as the original `model`.
# It outputs the embedding layer's output and the SimpleRNN layer's output (hidden states).
intermediate_model = Model(inputs=model.inputs, outputs=[model.get_layer('embed').output, model.get_layer('simple_rnn').output])

In [24]:
# Import SimpleRNN layer and alias it as SRNN for convenience.
from tensorflow.keras.layers import SimpleRNN as SRNN
# Define an input layer for the new inspection model, matching the shape and type of the original model's input.
seq_inp = Input(shape=(maxlen,), dtype='int32')
# Reuse the trained embedding layer from the original model to ensure consistent word vector representations.
seq_emb = model.get_layer('embed')(seq_inp)

# Create a new SimpleRNN layer. Crucially, `return_sequences=True` makes it output a hidden state for each timestep.
rnn_seq = SRNN(units=rnn_units, return_sequences=True, name='rnn_seq')

# Apply the new RNN layer to the embedded sequences to get the hidden states for each timestep.
# Keras automatically builds the layer when it's first called with input data.
seq_hidden = rnn_seq(seq_emb)

# Try to copy the trained weights from the original SimpleRNN layer to the new `rnn_seq` layer.
try:
    # Get the weights from the 'simple_rnn' layer of the pre-trained model.
    trained_weights = model.get_layer('simple_rnn').get_weights()
    # Set these weights to the `rnn_seq` layer, ensuring it behaves like the trained RNN.
    rnn_seq.set_weights(trained_weights)
    print("Copied RNN weights into sequence-inspection RNN.")
except Exception as e:
    print("Could not copy weights automatically:", e)

# Create the `inspect_model` which takes the input sequence and outputs the hidden states from `rnn_seq`.
inspect_model = Model(inputs=seq_inp, outputs=seq_hidden)

# Set the index of the sentence to inspect (e.g., the first sentence).
idx = 0
# Prepare the example sequence by selecting one padded sequence from `X` and ensuring it has a batch dimension.
example_seq = X[idx:idx+1]
# Use the `inspect_model` to predict the hidden states for each timestep of the `example_seq`.
hidden_seq = inspect_model.predict(example_seq)

# Print the original sentence for context.
print("Sentence:", sentences[idx])
# Print the token IDs of the example sequence.
print("Token ids:", example_seq)
# Print the shape of the resulting hidden states, indicating (batch_size, timesteps, rnn_units).
print("Hidden states per timestep shape:", hidden_seq.shape)
# Inform the user that the following output shows hidden states per timestep.
print("Hidden states (timesteps x units):")
# Print the hidden states, rounded to 3 decimal places for readability. `hidden_seq[0]` accesses the states for the first (and only) item in the batch.
print(np.round(hidden_seq[0], 3))

Copied RNN weights into sequence-inspection RNN.
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 163ms/step
Sentence: I love this product
Token ids: [[ 3 26  2  7  0]]
Hidden states per timestep shape: (1, 5, 8)
Hidden states (timesteps x units):
[[ 0.003  0.016 -0.006  0.024  0.067 -0.045 -0.001  0.091]
 [-0.073  0.16   0.109  0.173 -0.029 -0.113 -0.141 -0.181]
 [-0.362 -0.251 -0.317 -0.026  0.288 -0.157 -0.112  0.261]
 [ 0.328 -0.238  0.268 -0.259 -0.229  0.007  0.461 -0.325]
 [ 0.328 -0.238  0.268 -0.259 -0.229  0.007  0.461 -0.325]]


In [14]:
# Define an input layer for the new inspection model, matching the shape and type of the original model's input.
seq_inp = Input(shape=(maxlen,), dtype='int32')
# Reuse the trained embedding layer from the original model to ensure consistent word vector representations.
seq_emb = model.get_layer('embed')(seq_inp)

# Create a new SimpleRNN layer. Crucially, `return_sequences=True` makes it output a hidden state for each timestep.
rnn_seq = SRNN(units=rnn_units, return_sequences=True, name='rnn_seq')

# Apply the new RNN layer to the embedded sequences to get the hidden states for each timestep.
# Keras automatically builds the layer when it's first called with input data.
seq_hidden = rnn_seq(seq_emb)

# Try to copy the trained weights from the original SimpleRNN layer to the new `rnn_seq` layer.
try:
    # Get the weights from the 'simple_rnn' layer of the pre-trained model.
    trained_weights = model.get_layer('simple_rnn').get_weights()
    # Set these weights to the `rnn_seq` layer, ensuring it behaves like the trained RNN.
    rnn_seq.set_weights(trained_weights)
    print("Copied RNN weights into sequence-inspection RNN.")
except Exception as e:
    print("Could not copy weights automatically:", e)

# Create the `inspect_model` which takes the input sequence and outputs the hidden states from `rnn_seq`.
inspect_model = Model(inputs=seq_inp, outputs=seq_hidden)

# Set the index of the sentence to inspect (e.g., the first sentence).
idx = 0
# Prepare the example sequence by selecting one padded sequence from `X` and ensuring it has a batch dimension.
example_seq = X[idx:idx+1]
# Use the `inspect_model` to predict the hidden states for each timestep of the `example_seq`.
hidden_seq = inspect_model.predict(example_seq)

# Print the original sentence for context.
print("Sentence:", sentences[idx])
# Print the token IDs of the example sequence.
print("Token ids:", example_seq)
# Print the shape of the resulting hidden states, indicating (batch_size, timesteps, rnn_units).
print("Hidden states per timestep shape:", hidden_seq.shape)
# Inform the user that the following output shows hidden states per timestep.
print("Hidden states (timesteps x units):")
# Print the hidden states, rounded to 3 decimal places for readability. `hidden_seq[0]` accesses the states for the first (and only) item in the batch.
print(np.round(hidden_seq[0], 3))

Copied RNN weights into sequence-inspection RNN.
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 181ms/step
Sentence: I love this product
Token ids: [[ 3 26  2  7  0]]
Hidden states per timestep shape: (1, 5, 8)
Hidden states (timesteps x units):
[[-0.061 -0.071  0.019  0.006  0.03   0.022 -0.039 -0.049]
 [-0.046 -0.163 -0.035 -0.076  0.125 -0.186 -0.073 -0.345]
 [-0.306 -0.17  -0.241 -0.236  0.245 -0.076 -0.102 -0.284]
 [-0.311 -0.357 -0.185 -0.468  0.09  -0.061 -0.14  -0.499]
 [-0.311 -0.357 -0.185 -0.468  0.09  -0.061 -0.14  -0.499]]
